# Efficient FunSearch - Colab Reproduction Notebook

This notebook is designed for CS5491 milestone-style reproducibility and is aligned with:
- current `src/` interfaces
- `docs/milestone/reproduction.txt` output fields


## 1) Clone and install

In [ ]:
REPO_URL = 'https://github.com/jaycee6666/efficient-funsearch.git'
!git clone {REPO_URL}
%cd efficient-funsearch

In [ ]:
!pip install -q -e .
!pip install -q -e .[dev]

## 2) Sanity import

In [ ]:
from src.archive import ProgramArchive
from src.integration import FunSearchAdapter, FunSearchConfig
from src.integration.ablation_configs import get_v1_ablation_configs
from src.metrics import EfficiencyTracker
from src.normalizer import ProgramNormalizer
from src.similarity import HybridSimilarityDetector

print('Imports OK')

## 3) Core feature smoke checks

In [ ]:
normalizer = ProgramNormalizer()
detector = HybridSimilarityDetector()
archive = ProgramArchive()

code_a = "def solve(x):\n    return x + 1"
code_b = "def solve(y):\n    return y + 1"

norm_a = normalizer.normalize(code_a)
norm_b = normalizer.normalize(code_b)
sim = detector.is_similar(norm_a, norm_b)

archive.add(code_a, norm_a, score=1.0)
print('archive considers code_b duplicate:', archive.is_duplicate(norm_b))
print('similarity method:', sim.detection_method)

## 4) Proposal-v1 ablation variants

In [ ]:
[c.name for c in get_v1_ablation_configs()]

## 5) Milestone reproduction-style run

In [ ]:
import json


def eval_fn(code: str) -> float:
    return float(len(code))

initial_programs = [
    "def solve(x): return x + 1",
    "def solve(y): return y + 1",
    "def solve(x): return x + 1",
    "def solve(x): return x * 2",
]

def run(setting: str, use_dedup: bool):
    cfg = FunSearchConfig(max_generations=1, verbose=False, use_duplicate_detection=use_dedup)
    adapter = FunSearchAdapter(cfg)
    result = adapter.run(initial_programs=initial_programs, evaluation_function=eval_fn)
    return {
        "setting": setting,
        "N_total": result.total_programs_generated,
        "N_unique": result.unique_programs_evaluated,
        "sample_efficiency": result.efficiency_metrics.sample_efficiency,
        "duplicate_rate": result.efficiency_metrics.duplicate_detection_rate,
        "convergence_proxy": "1-generation smoke run",
        "final_quality_proxy": result.best_score,
    }

rows = [
    run("original", use_dedup=False),
    run("behavioral_plus_diversity", use_dedup=True),
]
print(json.dumps(rows, ensure_ascii=False, indent=2))

## 6) Optional verification commands

In [ ]:
!ruff check .
!pytest -q -rs

## 7) Tiny tracker demonstration

In [ ]:
tracker = EfficiencyTracker()
for i in range(6):
    tracker.record_generation()
    if i % 2 == 0:
        tracker.record_duplicate()
        tracker.record_filtered()
    else:
        tracker.record_evaluation()

m = tracker.metrics
print({
    'generated': m.total_programs_generated,
    'duplicates': m.duplicates_detected,
    'evaluated': m.programs_evaluated,
    'sample_efficiency': m.sample_efficiency,
})